# Tarea 2 · Optimización — ConRVP-SLA reducido con Gurobi

Modelo reducido de Campelo et al. (2019), implementado con `gurobipy` siguiendo la plantilla del tutorial de la ayudantía.

**Contenido**
1. Datos de la instancia
2. Modelo (variables, función objetivo y restricciones)
3. Resolver y ver las rutas
4. Experimentos
   - 4.1 Relajación continua y brecha de integralidad
   - 4.2 Variaciones de parámetros del solver
   - 4.3 Ajuste automático (`tune`)
   - 4.4 Análisis de sensibilidad de la capacidad Q

Nodos: `0` = depósito (salida), `1..n` = clientes, `n+1` = depósito (retorno). Los índices parten en 0 (día 1 = `t=0`, ruta 1 = `r=0`).

In [ ]:
import json
import math
import gurobipy as gp
from gurobipy import GRB

# Parámetros de las corridas (enunciado: Threads = 1 y TimeLimit = 600)
LIMITE = 600          # segundos por corrida
TIEMPO_TUNE = 1200    # segundos que se le da a tune()

## 1. Datos de la instancia

Se lee un archivo JSON con la instancia y se construyen las listas y matrices del modelo.
Para cambiar de instancia solo hay que cambiar `archivo`.

In [ ]:
import sys
sys.path.insert(0, 'codigo')
from modelo import leer_instancia

archivo = 'instancias/instancia_A.json'

d = leer_instancia(archivo)

# ---------- Conjuntos ----------
n, T, R = d['n'], d['T'], d['R']
N = range(n + 2)                # todos los nodos: 0, 1..n, n+1
Cl = range(1, n + 1)            # clientes
D = range(T)                    # días
Ru = range(R)                   # rutas

# ---------- Parámetros ----------
Q = d['Q']                      # capacidad del vehículo
apertura = d['apertura']        # a_W
cierre = d['cierre']            # b_W
dist = d['dist']
tv = d['tv']
q = d['q']
s = d['s']
a = d['a']
b = d['b']
con_ventana = d['con_ventana']
M = d['M']

print('Instancia:', d['nombre'], '| clientes =', n, '| días =', T, '| rutas =', R, '| Q =', Q)
print('Clientes con ventana de tiempo:', con_ventana)
print('M =', M)


## 2. Modelo

Variables:
- $X_{ijt}^r = 1$ si la ruta $r$ recorre el arco $(i,j)$ el día $t$
- $Z_{ir} = 1$ si el cliente $i$ pertenece a la ruta $r$ (el mismo cliente va siempre en la misma ruta)
- $W_{it}$: hora de llegada al cliente $i$ el día $t$
- $dW_r$: hora de salida de la ruta $r$ del depósito (igual todos los días)
- $rW_{rt}$: hora de retorno de la ruta $r$ el día $t$

$$\min \sum_{r}\sum_{t}\sum_{(i,j)\in A} d_{ij} X_{ijt}^r \qquad (1)$$

Las restricciones están numeradas como en el enunciado. El arco $(0,n+1)$ representa una ruta vacía ese día, para que la ecuación (5) sea factible.

In [ ]:
def armar_modelo():
    m = gp.Model('ConRVP-SLA')
    m.Params.OutputFlag = 0

    # Arcos permitidos: salir del depósito, entre clientes, volver al depósito
    A = [(0, j) for j in Cl] + [(0, n + 1)]
    A += [(i, j) for i in Cl for j in Cl if i != j]
    A += [(i, n + 1) for i in Cl]

    # ---------- Variables (16) ----------
    X = m.addVars([(r, i, j, t) for r in Ru for (i, j) in A for t in D], vtype=GRB.BINARY, name='X')
    Z = m.addVars(Cl, Ru, vtype=GRB.BINARY, name='Z')
    W = m.addVars(Cl, D, lb=apertura, ub=cierre, name='W')
    dW = m.addVars(Ru, name='dW')
    rW = m.addVars(Ru, D, name='rW')

    # ---------- Función objetivo (1) ----------
    m.setObjective(gp.quicksum(dist[i][j] * X[r, i, j, t] for r in Ru for (i, j) in A for t in D), GRB.MINIMIZE)

    # ---------- Restricciones ----------
    # (2) cada cliente pertenece a exactamente una ruta
    m.addConstrs((Z.sum(i, '*') == 1 for i in Cl), name='asignacion')

    # (3) si el cliente j está en la ruta r, entra un arco a j
    m.addConstrs((gp.quicksum(X[r, i, j, t] for i in N if (i, j) in A) == Z[j, r]
                  for j in Cl for r in Ru for t in D), name='entrada')

    # (4) si el cliente j está en la ruta r, sale un arco de j
    m.addConstrs((gp.quicksum(X[r, j, k, t] for k in N if (j, k) in A) == Z[j, r]
                  for j in Cl for r in Ru for t in D), name='salida')

    # (5) cada ruta sale una vez del depósito cada día
    m.addConstrs((gp.quicksum(X[r, 0, j, t] for j in N if (0, j) in A) == 1
                  for r in Ru for t in D), name='sale_deposito')

    # (7) tiempo de llegada entre dos clientes (Big-M)
    m.addConstrs((W[i, t] + s[i][t] + tv[i][j] <= W[j, t] + M * (1 - X[r, i, j, t])
                  for r in Ru for i in Cl for j in Cl if i != j for t in D), name='tiempo')

    # (8) ventanas de tiempo (solo clientes con ventana)
    m.addConstrs((W[i, t] >= a[i] for i in con_ventana for t in D), name='ventana_ini')
    m.addConstrs((W[i, t] <= b[i] for i in con_ventana for t in D), name='ventana_fin')

    # (9) salida del depósito -> primer cliente
    m.addConstrs((dW[r] + tv[0][j] <= W[j, t] + M * (1 - X[r, 0, j, t])
                  for r in Ru for j in Cl for t in D), name='salida_W')

    # (10) último cliente -> retorno al depósito
    m.addConstrs((W[i, t] + s[i][t] + tv[i][n + 1] <= rW[r, t] + M * (1 - X[r, i, n + 1, t])
                  for r in Ru for i in Cl for t in D), name='retorno_W')

    # ruta vacía: el retorno no puede ser antes de la salida
    m.addConstrs((dW[r] <= rW[r, t] + M * (1 - X[r, 0, n + 1, t])
                  for r in Ru for t in D), name='ruta_vacia')

    # (11) y (12) horario del depósito
    m.addConstrs((dW[r] >= apertura for r in Ru), name='abre')
    m.addConstrs((rW[r, t] <= cierre for r in Ru for t in D), name='cierra')

    # (14) capacidad del vehículo
    cap = m.addConstrs((gp.quicksum(q[i][t] * Z[i, r] for i in Cl) <= Q
                        for r in Ru for t in D), name='capacidad')

    return m, X, Z, W, cap


def mostrar_rutas(m, X, W):
    """Imprime la ruta de cada día siguiendo los arcos con X = 1."""
    for t in D:
        for r in Ru:
            actual = 0
            camino = []
            while actual != n + 1:
                for j in N:
                    if (r, actual, j, t) in X and X[r, actual, j, t].X > 0.5:
                        actual = j
                        break
                if actual != n + 1:
                    camino.append(f'{actual}(llega {W[actual, t].X:.0f})')
            carga = sum(q[int(c.split('(')[0])][t] for c in camino)
            texto = ' -> '.join(camino) if camino else 'vacía'
            print(f'Día {t+1} | Ruta {r+1} | carga {carga}/{Q} | D -> {texto} -> D')

## 3. Resolver

Corrida base: parámetros por defecto de Gurobi, con `Threads = 1` y `TimeLimit = LIMITE`.

In [ ]:
m, X, Z, W, cap = armar_modelo()
m.Params.Threads = 1
m.Params.TimeLimit = LIMITE
m.optimize()

if m.SolCount > 0:
    print(f'Distancia total = {m.ObjVal:.1f}')
    print(f'Estado = {m.Status} (2 = óptimo, 9 = límite de tiempo) | gap = {100*m.MIPGap:.2f} % | tiempo = {m.Runtime:.1f} s')
    print()
    mostrar_rutas(m, X, W)
else:
    print('Sin solución. Estado:', m.Status)

## 4.1 Relajación continua y brecha de integralidad

Se relajan las variables binarias a continuas ($0 \le X, Z \le 1$) con `m.relax()` y se compara con el óptimo entero:

$$\text{brecha} = \frac{z_{MILP} - z_{LP}}{z_{MILP}} \cdot 100$$

In [ ]:
z_MILP = m.ObjVal

relajado = m.relax()
relajado.optimize()
z_LP = relajado.ObjVal

brecha = (z_MILP - z_LP) / z_MILP * 100
print(f'z_MILP = {z_MILP:.2f}')
print(f'z_LP   = {z_LP:.2f}')
print(f'Brecha de integralidad = {brecha:.2f} %')

## 4.2 Variaciones de parámetros del solver

Se resuelve la misma instancia con distintas configuraciones y se compara tiempo, gap y valor objetivo.
Todas con `Threads = 1` y `TimeLimit = LIMITE`.

> Cada corrida puede demorar hasta `LIMITE` segundos (6 configuraciones × 600 s = 60 min como máximo).

In [ ]:
configuraciones = {
    'base':         {},
    'Cuts=0':       {'Cuts': 0},
    'Presolve=0':   {'Presolve': 0},
    'MIPFocus=1':   {'MIPFocus': 1},
    'MIPFocus=3':   {'MIPFocus': 3},
    'Heuristics=0': {'Heuristics': 0},
}

resultados = {}
for nombre, parametros in configuraciones.items():
    mc, Xc, Zc, Wc, capc = armar_modelo()
    mc.Params.Threads = 1
    mc.Params.TimeLimit = LIMITE
    for p, valor in parametros.items():
        mc.setParam(p, valor)
    mc.optimize()

    if mc.SolCount > 0:
        resultados[nombre] = (mc.Runtime, 100 * mc.MIPGap, mc.ObjVal, mc.NodeCount)
    else:
        resultados[nombre] = (mc.Runtime, None, None, mc.NodeCount)

print(f'{"configuración":15s} {"tiempo (s)":>11s} {"gap (%)":>9s} {"objetivo":>10s} {"nodos":>9s}')
for nombre, (tiempo, gap, obj, nodos) in resultados.items():
    gap_txt = '-' if gap is None else f'{gap:.2f}'
    obj_txt = '-' if obj is None else f'{obj:.1f}'
    print(f'{nombre:15s} {tiempo:11.1f} {gap_txt:>9s} {obj_txt:>10s} {nodos:9.0f}')

## 4.3 Ajuste automático de parámetros (`tune`)

`m.tune()` prueba distintas combinaciones de parámetros. Después se resuelve con la mejor encontrada.

In [ ]:
mt, Xt, Zt, Wt, capt = armar_modelo()
mt.Params.Threads = 1
mt.Params.TuneTimeLimit = TIEMPO_TUNE
mt.tune()

if mt.TuneResultCount > 0:
    mt.getTuneResult(0)                      # carga en el modelo los mejores parámetros
    mt.write('mejores_parametros.prm')       # los guarda en un archivo
    mt.Params.OutputFlag = 0
    mt.Params.Threads = 1
    mt.Params.TimeLimit = LIMITE
    mt.optimize()
    print(f'Con parámetros ajustados: objetivo = {mt.ObjVal:.1f} | gap = {100*mt.MIPGap:.2f} % | tiempo = {mt.Runtime:.1f} s')
    print('Parámetros que cambió tune (ver mejores_parametros.prm):')
    print(open('mejores_parametros.prm').read())
else:
    print('tune() no encontró una mejora respecto a los parámetros por defecto.')

## 4.4 Análisis de sensibilidad de la capacidad Q

Un MIP no tiene precios sombra, así que se usa `m.fixed()`: fija las variables enteras en la solución encontrada y deja un PL, donde sí se pueden leer
`Pi` (precio sombra), `Slack`, `SARHSLow` y `SARHSUp` (rango del lado derecho en que `Pi` sigue valiendo).

**Ojo:** como $X$ y $Z$ quedan fijas, el precio sombra de la capacidad sale 0 aunque la ruta esté llena, porque ayudaría a bajar la distancia solo si se pudieran reasignar clientes. Por eso después se re-optimiza el modelo completo con otros valores de Q.

In [ ]:
fijo = m.fixed()
fijo.Params.OutputFlag = 0
fijo.optimize()

print(f'{"restricción":18s} {"Pi":>6s} {"Slack":>8s} {"SARHSLow":>9s} {"SARHSUp":>9s}')
for r in Ru:
    for t in D:
        c = fijo.getConstrByName(f'capacidad[{r},{t}]')
        print(f'{c.ConstrName:18s} {c.Pi:6.2f} {c.Slack:8.1f} {c.SARHSLow:9.1f} {c.SARHSUp:9.1f}')

In [ ]:
# Re-optimizar el modelo completo con otros valores de Q
for factor in [0.9, 1.0, 1.1]:
    ms, Xs, Zs, Ws, caps = armar_modelo()
    ms.Params.Threads = 1
    ms.Params.TimeLimit = LIMITE
    for c in caps.values():
        c.RHS = Q * factor
    ms.optimize()
    if ms.SolCount > 0:
        print(f'Q = {Q*factor:7.1f} (x{factor}) -> distancia = {ms.ObjVal:.1f} | tiempo = {ms.Runtime:.1f} s')
    else:
        print(f'Q = {Q*factor:7.1f} (x{factor}) -> sin solución (estado {ms.Status}; 3 = infactible)')